In [5]:
"""
Assignment 1: Reinforcement Learning from Human Feedback (RLHF)
Notebook: 01_sft.ipynb

Purpose:
This notebook performs the Supervised Fine-Tuning (SFT) stage of the RLHF pipeline.
It starts from the base model Qwen/Qwen2.5-0.5B-Instruct and fine-tunes it on an
instruction-response dataset to produce a stronger instruction-following baseline model.

Summary of sections:
1. Environment setup and package imports
   - Imports required libraries and prepares the training environment.

2. Model and tokenizer loading
   - Loads the base language model and tokenizer.

3. Dataset loading and preprocessing
   - Loads the supervised instruction dataset and formats it into the structure
     required for fine-tuning.

4. SFT training configuration
   - Defines training arguments, PEFT/LoRA settings if used, and other parameters.

5. Model fine-tuning
   - Runs supervised fine-tuning on the prepared dataset.

6. Saving outputs
   - Saves the trained SFT model / adapter and tokenizer for later stages.

7. Sample generation / sanity check
   - Generates example outputs to confirm that the fine-tuned model works properly.

Main output:
- Fine-tuned SFT model used as the baseline model for reward modelling and PPO stages.
"""

'\nAssignment 1: Reinforcement Learning from Human Feedback (RLHF)\nNotebook: 01_sft.ipynb\n\nPurpose:\nThis notebook performs the Supervised Fine-Tuning (SFT) stage of the RLHF pipeline.\nIt starts from the base model Qwen/Qwen2.5-0.5B-Instruct and fine-tunes it on an\ninstruction-response dataset to produce a stronger instruction-following baseline model.\n\nSummary of sections:\n1. Environment setup and package imports\n   - Imports required libraries and prepares the training environment.\n\n2. Model and tokenizer loading\n   - Loads the base language model and tokenizer.\n\n3. Dataset loading and preprocessing\n   - Loads the supervised instruction dataset and formats it into the structure\n     required for fine-tuning.\n\n4. SFT training configuration\n   - Defines training arguments, PEFT/LoRA settings if used, and other parameters.\n\n5. Model fine-tuning\n   - Runs supervised fine-tuning on the prepared dataset.\n\n6. Saving outputs\n   - Saves the trained SFT model / adapter

In [6]:
# Install packages

!pip -q install transformers datasets trl peft accelerate bitsandbytes evaluate rouge_score


In [2]:
# Import Libraries

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


# Model and Configuration

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./outputs/sft_qwen25_05b_base"

SEED = 42
TRAIN_SAMPLES = 1000
EVAL_SAMPLES = 100
MAX_LENGTH = 512


# Load tokenizer and model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Model loaded successfully")
print("Pad token id:", tokenizer.pad_token_id)
print("EOS token id:", tokenizer.eos_token_id)


# Load dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

allowed_categories = ["open_qa", "brainstorming", "summarization"]
dataset = dataset.filter(lambda x: x["category"] in allowed_categories)
dataset = dataset.shuffle(seed=SEED)

train_dataset = dataset.select(range(min(TRAIN_SAMPLES, len(dataset))))
eval_start = len(train_dataset)
eval_end = min(eval_start + EVAL_SAMPLES, len(dataset))
eval_dataset = dataset.select(range(eval_start, eval_end))

print("Filtered total size:", len(dataset))
print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))
print(train_dataset[0])


# Convert to prompt-completion format

def format_dolly_prompt_completion(example):
    instruction = example["instruction"]
    context = example["context"] if example["context"] else ""
    response = example["response"]

    if context.strip():
        prompt = (
            f"Instruction: {instruction}\n\n"
            f"Context: {context}\n\n"
            f"Response:"
        )
    else:
        prompt = (
            f"Instruction: {instruction}\n\n"
            f"Response:"
        )

    completion = f" {response}"

    return {
        "prompt": prompt,
        "completion": completion
    }

train_dataset = train_dataset.map(format_dolly_prompt_completion)
eval_dataset = eval_dataset.map(format_dolly_prompt_completion)

print(train_dataset[0]["prompt"])
print(train_dataset[0]["completion"])


# LoRA configuration

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

print(peft_config)


# Training configuration

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    report_to="none",
    max_length=MAX_LENGTH,
    seed=SEED,
)


# Create trainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("Trainer created successfully")


# Train Model

trainer.train()


# Save model

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)


# Test generation

test_prompts = [
    "Explain RLHF in simple terms.",
    "What is a reward model in RLHF?",
    "Why is PPO used in RLHF?"
]

for prompt in test_prompts:
    text = f"Instruction: {prompt}\n\nResponse:"
    inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    print("=" * 80)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


# Compare base vs SFT

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

compare_prompts = [
    "Explain reinforcement learning.",
    "What is supervised fine-tuning?",
    "Why do we need human feedback in RLHF?"
]

for prompt in compare_prompts:
    text = f"Instruction: {prompt}\n\nResponse:"

    print("=" * 100)
    print("PROMPT:", prompt)

    base_inputs = tokenizer(text, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        base_outputs = base_model.generate(
            **base_inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    print("\n--- Base Model Output ---")
    print(tokenizer.decode(base_outputs[0], skip_special_tokens=True))

    sft_inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        sft_outputs = trainer.model.generate(
            **sft_inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    print("\n--- SFT Model Output ---")
    print(tokenizer.decode(sft_outputs[0], skip_special_tokens=True))



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Step,Training Loss,Validation Loss
50,2.351905,2.575719
100,2.254070,2.555428


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Saved to: ./outputs/sft_qwen25_05b_base
Instruction: Explain RLHF in simple terms.

Response: R/T/T/T/API/API/API
  1、     

    // code-block-form-block"> 
 

 

   �、“2000) =    "3. 《国网\/DIVANGARspec/spec- The problem structure/database/API/API/*****
   �、“代码 syntax, 问题
  [2、 �

...

)

    var json.log.clear-align align�

  5、建设社会主义建设大楼的数据显示数据显示数据显示：  2、  个人结构力学科学证明题） 


  海


Instruction: What is a reward model in RLHF?

Response: In the code/API/API/API/*****
  2、《关于对给定
    var.clear/views/services/API/API/T/T/T/H/T/Tspec/spec/API/API API/API/API/F/T/T\//T/TAPI/API/API/api/API/API

                  
          code syntax, "001. The problem structure/database/API/API/logout/API/API"> 

 
   �、“代码/API/API/ui/API/API Code/API/API/UI/API/API/problems.log.log.log.js/API/API/action-box-form-block- H) = { 
Instruction: Why is PPO used in RLHF?

Response: The problem structure

 《关于对给定/API/API/API/ / code syntax, "2、 �、“1.    �、     
    var.clear-align-block"> 

 
    问题
  000) =   [3、建

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT: Explain reinforcement learning.

--- Base Model Output ---
Instruction: Explain reinforcement learning.

Response: Reinforcement learning is a type of machine learning where an agent learns to make decisions by receiving rewards or punishments in response to its actions. The goal of the agent is to maximize its cumulative reward over time, which can be achieved through trial and error or through exploration and exploitation strategies.
In reinforcement learning, the environment provides feedback on the performance of the agents, such as how well they solve problems or achieve certain goals. The agent then uses this feedback to update its behavior and learn from experience. The process

--- SFT Model Output ---
Instruction: Explain reinforcement learning.

Response: Rein/API/API/API/*****
   �、“000. The problem

     
    var code-block-form/API/API/T/T/T2、建设现代化的数据显示,   《关于对给定代码 syntax)

...



   "1、   �、“  问题： 

var React API structure/database/views/services/API/APIAPI/API/AP

In [3]:
# Save model
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_root = "/content/drive/MyDrive/rlhf_assignment_outputs"
os.makedirs(drive_root, exist_ok=True)

target_path = os.path.join(drive_root, "sft_qwen25_05b_base")

if os.path.exists(target_path):
    shutil.rmtree(target_path)

shutil.copytree(OUTPUT_DIR, target_path)

print("Copied SFT adapter to Drive:", target_path)
print("Files in Drive copy:")
for f in os.listdir(target_path):
    print(" ", f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied SFT adapter to Drive: /content/drive/MyDrive/rlhf_assignment_outputs/sft_qwen25_05b_base
Files in Drive copy:
  chat_template.jinja
  checkpoint-100
  tokenizer.json
  adapter_model.safetensors
  tokenizer_config.json
  adapter_config.json
  training_args.bin
  README.md
  checkpoint-125
